# 1. 流式调用（stream）

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 加载配置文件
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL')

# 获取大模型
model = init_chat_model(
    model_provider='deepseek',
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

for chunk in model.stream('写一首七言律诗，总结大模型的发展'):
    print(chunk.text, end="", flush=True) # 逐token输出

# 2. 批量调用（batch）

In [ ]:
messages = ['1 + 5 = ?', '奥特曼是真的吗?']
responses = model.batch(messages)
for response in responses:
    print(response)

# 3. 按照完成的顺序 接收响应（batch_as_completed）

In [ ]:
messages = ['你好，你是谁？','2 + 3 * 5 = ?','中国首都在哪里？']
responses = model.batch_as_completed(messages)
for response in responses:
    print(response)

# 4. 性能对比
## 使用batch():

In [14]:
import time
# 准备多个输入
inputs = [
    '翻译成英文：春天来了',
    '翻译成英文：夏天很热',
    '翻译成英文：秋天落叶',
    '翻译成英文：冬天下雪'
]
# 批量调用（高效）
start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start
print('批量调用结果：')
for i, response in enumerate(responses):
    print(f'{i+1}. {response.content}')
print(f'耗时: {batch_time:.2f}秒\n')

批量调用结果：
1. Spring has arrived.
2. Summer is very hot.
3. Autumn fallen leaves.
4. It snows in winter.
耗时: 1.25秒



## 使用循环调用invoke():

In [16]:
# 循环调用（低效，仅用于对比）
inputs = [
    '翻译成英文：春天来了',
    '翻译成英文：夏天很热',
    '翻译成英文：秋天落叶',
    '翻译成英文：冬天下雪'
]
# 开始时间
start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
# 结束时间
loop_time = time.time() - start
for i,response in enumerate(responses):
    print(f'{i+1}. {response.content}')
print(f'循环调用耗时: {loop_time: .2f}秒')
print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. Spring has arrived.
2. Summer is very hot.
3. Autumn fallen leaves.
4. It snows in winter.
循环调用耗时:  5.51秒
批量调用节省: 77.3%
